In [ ]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.ensemble import HistGradientBoostingRegressor

from skopt import gp_minimize
from skopt.space import Real

In [ ]:
# =====================================================
# 1. 데이터 불러오기
# =====================================================

DATA_PATH = Path("film_experiments.csv")

df = pd.read_csv(DATA_PATH)

print("데이터 크기:", df.shape)
print(df.head())
print(df.info())


: 

In [ ]:
# 필름 배합·공정 조건 → 물성 예측 & 베이지안 최적화

**입력 (7):** resin_A_ratio, resin_B_ratio, plasticizer, compatibilizer, uv_stabilizer, process_temp, draw_ratio  
**타깃 (3):** tensile_strength, transparency, barrier  

조성 4변수는 합≈100, 물성 3개는 트레이드오프 → **목적함수(가중치)를 먼저 정의**한 뒤, 학습한 회귀 모델을 대리모델로 베이지안 최적화합니다.

In [ ]:
# =====================================================
# 2. 탐색적 분석 — 조성 제약 & 물성 트레이드오프
# =====================================================

FEATURES = [
    "resin_A_ratio", "resin_B_ratio", "plasticizer", "compatibilizer",
    "uv_stabilizer", "process_temp", "draw_ratio",
]
TARGETS = ["tensile_strength", "transparency", "barrier"]

comp_sum = df[FEATURES[:4]].sum(axis=1)
print(f"조성 합: mean={comp_sum.mean():.4f}, std={comp_sum.std():.4f}")

corr_targets = df[TARGETS].corr()
print("\n물성 상관계수:")
display(corr_targets.round(3))

fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))
pairs = [
    ("tensile_strength", "transparency"),
    ("tensile_strength", "barrier"),
    ("transparency", "barrier"),
]
for ax, (x, y) in zip(axes, pairs):
    ax.scatter(df[x], df[y], alpha=0.35, s=18)
    ax.set_xlabel(x)
    ax.set_ylabel(y)
    r = df[x].corr(df[y])
    ax.set_title(f"{x} vs {y}  (r={r:.2f})")
plt.tight_layout()
plt.show()

## 목적함수 정의

물성 3개를 동시에 최대화할 수 없으므로 **정규화 후 가중합**으로 스칼라 목적함수를 만듭니다.

| 물성 | 다른 물성과의 관계 | 해석 |
|------|-------------------|------|
| tensile_strength ↔ transparency | r ≈ -0.64 | 강도↑ → 투명도↓ |
| transparency ↔ barrier | r ≈ -0.72 | 투명도↑ → 차단성↓ |
| tensile_strength ↔ barrier | r ≈ +0.58 | 강도↑ → 차단성↑ |

아래 `WEIGHTS`를 바꿔 **무엇을 우선할지** 결정합니다. (합=1)

In [ ]:
# =====================================================
# 3. 대리모델(회귀) 학습
# =====================================================

X = df[FEATURES].values
y = df[TARGETS].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

surrogate = MultiOutputRegressor(
    RandomForestRegressor(
        n_estimators=300,
        max_depth=12,
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1,
    )
)
surrogate.fit(X_train, y_train)

y_pred = surrogate.predict(X_test)

print("=== 테스트 성능 (R² / RMSE / MAE) ===")
for i, name in enumerate(TARGETS):
    r2 = r2_score(y_test[:, i], y_pred[:, i])
    rmse = mean_squared_error(y_test[:, i], y_pred[:, i], squared=False)
    mae = mean_absolute_error(y_test[:, i], y_pred[:, i])
    print(f"{name:18s}  R²={r2:.3f}  RMSE={rmse:.2f}  MAE={mae:.2f}")

In [ ]:
# =====================================================
# 4. 목적함수 & 탐색 공간 정의
# =====================================================

# --- 우선순위 설정 (합=1) ---
# 예) 차단성 우선 포장필름: [0.2, 0.2, 0.6]
#     투명 필름:            [0.2, 0.6, 0.2]
#     기계적 강도 우선:     [0.6, 0.2, 0.2]
WEIGHTS = np.array([1/3, 1/3, 1/3])

y_min = y_train.min(axis=0)
y_max = y_train.max(axis=0)
y_range = np.where(y_max - y_min == 0, 1.0, y_max - y_min)


def build_feature_vector(resin_A, resin_B, plasticizer, uv, temp, draw):
    """조성 4변수 합=100 제약을 compatibilizer로 흡수."""
    compat = 100.0 - resin_A - resin_B - plasticizer
    return np.array([[resin_A, resin_B, plasticizer, compat, uv, temp, draw]])


def predict_properties(x):
    resin_A, resin_B, plasticizer, uv, temp, draw = x
    X_new = build_feature_vector(resin_A, resin_B, plasticizer, uv, temp, draw)
    return surrogate.predict(X_new)[0]


def scalar_score(props):
    norm = (props - y_min) / y_range
    return float(np.dot(norm, WEIGHTS))


def bo_objective(params):
    """skopt는 최소화 → 음의 스칼라 점수 반환."""
    props = predict_properties(params)
    compat = 100.0 - params[0] - params[1] - params[2]

    penalty = 0.0
    if compat < 0 or compat > df["compatibilizer"].max():
        penalty += 10.0
    if props.min() < y_min.min() - 5 or props.max() > y_max.max() + 5:
        penalty += 5.0

    return -(scalar_score(props) - penalty)


# 자유 변수 6개 (compatibilizer는 잔여량)
search_space = [
    Real(df["resin_A_ratio"].min(), df["resin_A_ratio"].max(), name="resin_A_ratio"),
    Real(df["resin_B_ratio"].min(), df["resin_B_ratio"].max(), name="resin_B_ratio"),
    Real(df["plasticizer"].min(), df["plasticizer"].max(), name="plasticizer"),
    Real(df["uv_stabilizer"].min(), df["uv_stabilizer"].max(), name="uv_stabilizer"),
    Real(df["process_temp"].min(), df["process_temp"].max(), name="process_temp"),
    Real(df["draw_ratio"].min(), df["draw_ratio"].max(), name="draw_ratio"),
]

print("목적함수 가중치:", dict(zip(TARGETS, WEIGHTS.round(3))))

In [ ]:
# =====================================================
# 5. 베이지안 최적화 (대리모델로 목적함수 평가)
# =====================================================

# 기존 실험 중 최고점 (베이스라인)
baseline_scores = np.array([scalar_score(row) for row in y])
best_idx = baseline_scores.argmax()
print("기존 실험 최고점 (정규화 가중합):", baseline_scores[best_idx].round(4))
print("해당 조건:")
display(df.loc[[best_idx], FEATURES + TARGETS])

result = gp_minimize(
    func=bo_objective,
    dimensions=search_space,
    n_calls=120,
    n_initial_points=30,
    random_state=42,
    acq_func="EI",
    verbose=False,
)

best_params = result.x
best_props = predict_properties(best_params)
compat = 100.0 - best_params[0] - best_params[1] - best_params[2]

optimal = {
    "resin_A_ratio": best_params[0],
    "resin_B_ratio": best_params[1],
    "plasticizer": best_params[2],
    "compatibilizer": compat,
    "uv_stabilizer": best_params[3],
    "process_temp": best_params[4],
    "draw_ratio": best_params[5],
}
optimal.update(dict(zip(TARGETS, best_props)))

print("\n=== 베이지안 최적화 추천 조건 (대리모델 예측) ===")
for k, v in optimal.items():
    print(f"  {k:18s}: {v:.3f}" if isinstance(v, float) else f"  {k}: {v}")

print(f"\n예측 스칼라 점수: {scalar_score(best_props):.4f}")
print(f"기존 최고 대비 개선: {scalar_score(best_props) - baseline_scores[best_idx]:+.4f}")